# 09 Re-evaluation: evaluation

Needs notebooks 05 to 07. Runs on CPU. Contents:
1. Choice of the strongest CNN baseline on select days, before any test score.
2. CRPS, quantile loss, RMSE, MAE, and coverage and width of 50, 80, 90 and 95% intervals, before and after split and adaptive conformal calibration on calib days.
3. Paired day-level bootstrap (2,000 resamples).
4. Results by weather and on the broad test set.
5. Reserve sizing with smart persistence as the common schedule, at 90, 95 and 97.5% reliability.
6. Reliability diagram, example day and per-forecast table.

All outputs are saved in `MyDrive/UROP/review/`.

In [ ]:
!pip install -q datasets pvlib
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/UROP'
LOCAL_CACHE = '/content/skippd_cache'

In [ ]:
import os, shutil, time, math, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
P_DIR = f'{OUT}/review'
for d in [P_DIR, f'{P_DIR}/ckpt', f'{P_DIR}/preds', f'{P_DIR}/figs', f'{OUT}/cache', LOCAL_CACHE]:
    os.makedirs(d, exist_ok=True)
SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
INPUT_LEN, HORIZON = 15, 15
LAT, LON, ALT = 37.4275, -122.1697, 23
PANEL_TILT, PANEL_AZIMUTH, PANEL_KW_DC = 22.5, 195, 30.1
PV_CAP_PCT = 99.95
SEL_C = dict(geo=True, cs=True, kout=True, var=True, resid=True)
TRAIN = dict(batch=256, lr=1e-3, max_epochs=25, patience=4)
DIFF = dict(T=100, max_epochs=60, patience=6, lr=2e-4, batch=512, n_samples=50)
SPLIT_FRAC = dict(train=0.70, select=0.15, calib=0.15)
TIME_TOL_S = 1.0
SPIKE_KW = 8.0
QUANTILES = [0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975]

In [ ]:
SEEDS = [0, 1, 2]
LEVELS = [0.5, 0.8, 0.9, 0.95]
RELIAB = [0.90, 0.95, 0.975]
N_BOOT = 2000
VAR_BIN_QUANTILES = [0.4, 0.7, 0.9]

## 1. Data, windows and splits

In [ ]:
def load_split(split):
    names = ['img', 'pv', 'ts']
    local = {k: f'{LOCAL_CACHE}/{split}_{k}.npy' for k in names}
    drive_ = {k: f'{OUT}/cache/{split}_{k}.npy' for k in names}
    if not all(os.path.exists(p) for p in local.values()) and all(os.path.exists(p) for p in drive_.values()):
        print(f'[{split}] copying cache from Drive to local disk...')
        for k in names: shutil.copy(drive_[k], local[k])
    if all(os.path.exists(p) for p in local.values()):
        print(f'[{split}] loading from cache')
        img = np.load(local['img']); pv = np.load(local['pv']); ts_raw = np.load(local['ts'])
    else:
        from datasets import load_dataset
        print(f'[{split}] downloading from Hugging Face (first run only)...')
        ds = load_dataset('solarbench/SKIPPD', split=split)
        n = len(ds); img = np.empty((n, 64, 64, 3), dtype=np.uint8); pos = 0
        for b in ds.iter(batch_size=2000):
            arr = np.stack([np.asarray(im.convert('RGB')) for im in b['image']]); img[pos:pos + len(arr)] = arr; pos += len(arr)
        pv = np.asarray(ds['pv'], dtype=np.float32)
        t = pd.to_datetime(ds['time'])
        if t.tz is not None: t = t.tz_convert('UTC').tz_localize(None)
        ts_raw = t.values.astype('datetime64[ns]')
        for k, v in [('img', img), ('pv', pv), ('ts', ts_raw)]:
            np.save(local[k], v); shutil.copy(local[k], drive_[k])
    order = np.argsort(ts_raw, kind='stable')
    return dict(name=split, img=img, img_idx=order, pv=pv[order].astype(np.float32), ts=pd.DatetimeIndex(ts_raw[order]))

In [ ]:
TR = load_split('train'); TE = load_split('test')
P = np.load(f'{OUT}/day1b_prep.npz', allow_pickle=True)
TR['good'], TE['good'] = P['tr_good'].astype(bool), P['te_good'].astype(bool)
print('train rows', len(TR['pv']), '| test rows', len(TE['pv']))

In [ ]:
from pvlib.location import Location
from pvlib import irradiance
SITE = Location(LAT, LON, altitude=ALT)
def solar_terms(ts_naive):
    a = pd.DatetimeIndex(ts_naive).tz_localize('UTC')
    cs = SITE.get_clearsky(a, model='ineichen'); sp = SITE.get_solarposition(a)
    poa = irradiance.get_total_irradiance(PANEL_TILT, PANEL_AZIMUTH, dni=cs['dni'], ghi=cs['ghi'], dhi=cs['dhi'],
                                          solar_zenith=sp['apparent_zenith'], solar_azimuth=sp['azimuth'])
    return (np.clip(PANEL_KW_DC * poa['poa_global'].values / 1000.0, 0, PANEL_KW_DC).astype(np.float32),
            sp['apparent_zenith'].values.astype(np.float32), sp['azimuth'].values.astype(np.float32))
def local_date(ts): return np.array((pd.DatetimeIndex(ts) - pd.Timedelta(hours=8)).date)
def is_dst(d): return bool((pd.Timestamp(d) + pd.Timedelta(hours=12)).tz_localize('America/Los_Angeles').dst())
CLOCK_FIX = json.load(open(f'{OUT}/clock_fix.json')); print('clock fix:', CLOCK_FIX)
def solar_time(ts):
    ts = pd.DatetimeIndex(ts); ld = local_date(ts); lut = {d: is_dst(d) for d in set(ld)}
    shift = np.where(np.array([lut[d] for d in ld]), CLOCK_FIX['dst_min'], CLOCK_FIX['std_min'])
    return ts - pd.to_timedelta(shift, unit='min')
for D in (TR, TE):
    cs, zen, azi = solar_terms(solar_time(D['ts']))
    D['cs_raw'] = np.nan_to_num(cs); D['zen'] = np.nan_to_num(zen, nan=180.0); D['azi'] = np.nan_to_num(azi)
    D['ld'] = local_date(D['ts'])
    doy = D['ts'].dayofyear.values
    D['doy_s'] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32); D['doy_c'] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

In [ ]:
def spike_flags(pv):
    d1 = np.r_[0, np.diff(pv)]; d2 = np.r_[np.diff(pv), 0]
    return (np.abs(d1) > SPIKE_KW) & (np.abs(d2) > SPIKE_KW) & (np.sign(d1) != np.sign(d2))

def build_windows(D, good, exact=True):
    """Windows of INPUT_LEN readings plus a target HORIZON readings later.
       exact=True: every step must be 60 s +/- TIME_TOL_S (so the target is exactly 15 min after issue).
       exact=False: the notebook 01 rule (no step > 1 min, no step <= 0)."""
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    d = np.diff(s)
    bad = (np.abs(d - 60.0) > TIME_TOL_S) if exact else ((d > 60.0 + 1e-6) | (d <= 0))
    cb = np.r_[0, np.cumsum(bad)]
    t = np.arange(INPUT_LEN - 1, len(s) - HORIZON); st, en = t - (INPUT_LEN - 1), t + HORIZON
    ok = (cb[en] - cb[st]) == 0
    cg = np.r_[0, np.cumsum(~good)]; ok &= (cg[en + 1] - cg[st]) == 0
    if exact:
        ok &= np.abs(s[t + HORIZON] - s[t] - 60.0 * HORIZON) <= TIME_TOL_S
    return t[ok]

def other_pv_flags(D, min_elev=5.0, stuck_n=10, cs_ratio=1.6):
    """Quality checks from notebook 01 other than the spike rule: negative, above the rating,
       above cs_ratio x clear-sky, stuck readings in daylight, low sun."""
    pv, cs, zen = D['pv'], D['cs_raw'], D['zen']
    day = zen < 90 - min_elev
    run_id = np.r_[0, np.cumsum(np.diff(pv) != 0)]
    run_len = np.bincount(run_id)[run_id]
    stuck = (run_len >= stuck_n) & day & (pv > 0)
    return (pv < 0) | (pv > PANEL_KW_DC) | ((pv > cs_ratio * cs) & day) | stuck | ~day

def broad_mask(D):
    """Broad set: restore ONLY readings whose sole problem is the spike rule."""
    return D['good'] | (D['spike'] & ~other_pv_flags(D))

def horizon_seconds(D, w):
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    return s[w + HORIZON] - s[w]

In [ ]:
Z = np.load(f'{P_DIR}/windows_and_splits.npz', allow_pickle=True)
W = {k[2:]: Z[k] for k in Z.files if k.startswith('w_')}
DAY_SPLIT = dict(zip(pd.to_datetime(Z['tr_days']).date, Z['tr_day_split']))
TR['split'] = np.array([DAY_SPLIT[d] for d in TR['ld']])
PV_CAP = float(Z['pv_cap'])
for D in (TR, TE):
    D['cs'] = np.minimum(D['cs_raw'], PV_CAP).astype(np.float32)
    D['k'] = np.clip(D['pv'] / np.maximum(D['cs'], 1.0), 0, 1.5).astype(np.float32)
    D['spike'] = spike_flags(D['pv'])
for k, v in W.items(): print(f'{k:>10}: {len(v)} windows')
print(f'PV cap (training days): {PV_CAP:.2f} kW')

In [ ]:
MODEL_VERSIONS = dict(cnn_mse='v1', cnn_gauss='v1', cnn_quant='v2-sorted', diffusion='v1')
def experiment_id():
    h = hashlib.sha256()
    for k in sorted(W): h.update(k.encode()); h.update(np.ascontiguousarray(W[k]).astype(np.int64).tobytes())
    cfg = dict(TRAIN=TRAIN, DIFF=DIFF, SPLIT_FRAC=SPLIT_FRAC, TIME_TOL_S=TIME_TOL_S, SPIKE_KW=SPIKE_KW, QUANTILES=QUANTILES,
               INPUT_LEN=INPUT_LEN, HORIZON=HORIZON, PV_CAP=round(float(PV_CAP), 4), MODEL_VERSIONS=MODEL_VERSIONS,
               DAY_SPLIT={str(d): str(v) for d, v in sorted(DAY_SPLIT.items())})
    h.update(json.dumps(cfg, sort_keys=True).encode()); return h.hexdigest()[:16]
EXP_ID = experiment_id(); _ep = f'{P_DIR}/experiment.json'
def _write_exp(): json.dump(dict(exp_id=EXP_ID, written=pd.Timestamp.now().isoformat(timespec='seconds'), model_versions=MODEL_VERSIONS), open(_ep, 'w'), indent=2)
if globals().get('MIGRATE', False):
    _write_exp(); print('experiment.json written by migration (notebook 08)')
elif os.path.exists(_ep):
    _old = json.load(open(_ep))['exp_id']
    if _old != EXP_ID:
        raise SystemExit(f'this notebook builds experiment {EXP_ID}, but review/experiment.json is {_old}. '
                         'Windows, splits, settings or model versions differ from the saved run.')
elif globals().get('CREATE_EXPERIMENT', False):
    _write_exp()
else:
    raise SystemExit('review/experiment.json not found. Run notebook 05 (new experiment) or notebook 08 (existing results) first.')
print('experiment id:', EXP_ID)

In [ ]:
def _pred_path(name, seed): return f'{P_DIR}/preds/{name}_seed{seed}.npz'
def pred_exp_id(name, seed):
    return json.loads(str(np.load(_pred_path(name, seed), allow_pickle=True)['meta'])).get('exp_id')
def save_preds(name, seed, arrays, meta=None):
    p = _pred_path(name, seed)
    np.savez_compressed(p, **arrays, meta=json.dumps({**(meta or {}), 'exp_id': EXP_ID}))
    return p
def have_preds(name, seed):
    if not os.path.exists(_pred_path(name, seed)): return False
    e = pred_exp_id(name, seed)
    if e != EXP_ID: raise SystemExit(f'{name} seed {seed}: saved for experiment {e}, current is {EXP_ID}. Move or delete it, or run notebook 08.')
    return True
def load_preds(name, seed):
    assert have_preds(name, seed), f'missing predictions: {name} seed {seed}'
    return dict(np.load(_pred_path(name, seed), allow_pickle=True))
EVAL_SETS = ['select', 'calib', 'test', 'test_broad']
def data_of(k): return TE if k.startswith('test') else TR

In [ ]:
def smart_base(D, t):
    return np.clip(D['pv'][t] / np.maximum(D['cs_raw'][t], 1e-3) * D['cs_raw'][t + HORIZON], 0, PV_CAP).astype(np.float32)
def data_of(k): return TE if k.startswith('test') else TR
PROTOCOL = json.load(open(f'{P_DIR}/protocol.json')); print('protocol frozen on', PROTOCOL['written'])
Y = {k: data_of(k)['pv'][W[k] + HORIZON].astype(np.float64) for k in ['select', 'calib', 'test', 'test_broad']}
DAYS = {k: data_of(k)['ld'][W[k]] for k in Y}
OFF = np.arange(-(INPUT_LEN - 1), 1)
RV = {k: data_of(k)['k'][W[k][:, None] + OFF].std(1) for k in Y}

## 2. Forecasts and scores

In [ ]:
from scipy.stats import norm
import glob
QL = np.array(QUANTILES)
def F_point(mu): return dict(kind='point', mu=np.asarray(mu, float))
def F_gauss(mu, sd): return dict(kind='gauss', mu=np.asarray(mu, float), sd=np.maximum(np.asarray(sd, float), 1e-3))
def F_mix(mus, sds): return dict(kind='mix', mus=np.asarray(mus, float), sds=np.maximum(np.asarray(sds, float), 1e-3))
def F_samples(S): return dict(kind='samples', S=np.sort(np.asarray(S, float), 1))
def F_quant(Q): return dict(kind='quant', Q=np.sort(np.asarray(Q, float), 1))

def point(f):
    return {'point': lambda: f['mu'], 'gauss': lambda: f['mu'], 'mix': lambda: f['mus'].mean(1),
            'samples': lambda: f['S'].mean(1), 'quant': lambda: f['Q'][:, len(QL) // 2]}[f['kind']]()

def mix_ppf(mus, sds, p, it=60):
    lo = (mus - 8 * sds).min(1); hi = (mus + 8 * sds).max(1)
    for _ in range(it):
        mid = (lo + hi) / 2; c = norm.cdf((mid[:, None] - mus) / sds).mean(1)
        lo, hi = np.where(c < p, mid, lo), np.where(c < p, hi, mid)
    return (lo + hi) / 2

def quantile(f, p):
    k = f['kind']
    if k == 'point': return f['mu']
    if k == 'gauss': return f['mu'] + norm.ppf(p) * f['sd']
    if k == 'mix': return mix_ppf(f['mus'], f['sds'], p)
    if k == 'samples': return np.quantile(f['S'], p, axis=1)
    i = np.where(np.isclose(QL, p))[0]; assert len(i), f'quantile {p} not predicted by the quantile CNN'
    return f['Q'][:, i[0]]

def _A(m, v):
    s = np.sqrt(v); z = m / s; return 2 * s * norm.pdf(z) + m * (2 * norm.cdf(z) - 1)
def crps_w(f, y):
    '''Per-window CRPS in kW (None for the quantile CNN).'''
    k = f['kind']
    if k == 'point': return np.abs(f['mu'] - y)
    if k == 'gauss': return _A(y - f['mu'], f['sd'] ** 2) - f['sd'] / np.sqrt(np.pi)
    if k == 'mix':
        mus, v = f['mus'], f['sds'] ** 2; K = mus.shape[1]
        t1 = _A(y[:, None] - mus, v).mean(1)
        t2 = sum(_A(mus[:, i] - mus[:, j], v[:, i] + v[:, j]) for i in range(K) for j in range(K)) / K ** 2
        return t1 - 0.5 * t2
    if k == 'samples':
        S = f['S']; m = S.shape[1]; w = (2 * np.arange(1, m + 1) - m - 1) / m ** 2
        return np.abs(S - y[:, None]).mean(1) - (S * w).sum(1)
    return None

def qloss_w(f, y):
    '''Twice the mean pinball loss over the nine quantile levels, in kW.'''
    return 2 * np.mean([np.maximum(p * (y - quantile(f, p)), (p - 1) * (y - quantile(f, p))) for p in QL], 0)

def conf_q(s, level):
    n = len(s); return float(np.quantile(s, min(1.0, math.ceil((n + 1) * level) / n)))

def bins_of(rv, edges): return np.digitize(rv, edges)
EDGES = np.quantile(RV['calib'], VAR_BIN_QUANTILES)

def interval(f, fc, level, target, calib='none'):
    '''Central interval on `target`. f = forecast on target, fc = the same method on calib days.
       calib: 'none' (raw), 'split' (one conformal group), 'adaptive' (by recent variability).'''
    a = (1 - level) / 2
    lo, hi = quantile(f, a), quantile(f, 1 - a)
    if calib == 'none':
        return (lo, hi) if f['kind'] != 'point' else (np.full_like(lo, np.nan), np.full_like(lo, np.nan))
    clo, chi = quantile(fc, a), quantile(fc, 1 - a); sc = np.maximum(clo - Y['calib'], Y['calib'] - chi)
    bc = bins_of(RV['calib'], EDGES) if calib == 'adaptive' else np.zeros(len(sc), int)
    bt = bins_of(RV[target], EDGES) if calib == 'adaptive' else np.zeros(len(lo), int)
    q = np.zeros(len(lo))
    for b in np.unique(bt): q[bt == b] = conf_q(sc[bc == b], level)
    return lo - q, hi + q

def lower_bound(f, fc, rel, target):
    '''Lower bound expected to be exceeded with probability `rel`, adaptively calibrated on calib days.'''
    L, Lc = quantile(f, 1 - rel), quantile(fc, 1 - rel); sc = Lc - Y['calib']
    bc, bt = bins_of(RV['calib'], EDGES), bins_of(RV[target], EDGES); q = np.zeros(len(L))
    for b in np.unique(bt): q[bt == b] = conf_q(sc[bc == b], rel)
    return np.maximum(L - q, 0)

## 3. Load all methods

In [ ]:
def sp_forecast(k): return F_point(smart_base(data_of(k), W[k]))
def stack(name, k, seeds=SEEDS):
    '''Load every seed of a model; stops if a seed is missing.'''
    missing = [s for s in seeds if not have_preds(name, s)]
    if missing: raise SystemExit(f'{name} is missing seeds {missing}; expected {list(seeds)}.')
    return [load_preds(name, s)[k] for s in seeds]
SEED_TAG = f'{len(SEEDS)} seeds'
assert SEED_TAG == '3 seeds', 'method labels below assume 3 seeds'

def build_methods(k):
    M = {'smart persistence': sp_forecast(k)}
    a = stack('cnn_mse', k)
    for i, x in enumerate(a): M[f'MSE CNN, seed {i}'] = F_point(x[:, 0])
    if a: A = np.stack([x[:, 0] for x in a], 1); M['MSE CNN, 3 seeds (spread)'] = F_gauss(A.mean(1), A.std(1))
    a = stack('cnn_gauss', k)
    for i, x in enumerate(a): M[f'Gaussian CNN, seed {i}'] = F_gauss(x[:, 0], x[:, 1])
    if a: M['Gaussian CNN mixture, 3 seeds'] = F_mix(np.stack([x[:, 0] for x in a], 1), np.stack([x[:, 1] for x in a], 1))
    a = stack('cnn_quant', k)
    for i, x in enumerate(a): M[f'quantile CNN, seed {i}'] = F_quant(x)
    if a: M['quantile CNN, 3 seeds averaged'] = F_quant(np.mean(a, 0))
    for name, lab in [('diff', 'diffusion, image'), ('diff_noimg', 'diffusion, no image')]:
        a = stack(name, k)
        for i, x in enumerate(a): M[f'{lab}, seed {i}'] = F_samples(x)
        if a: M[f'{lab}, 3 seeds pooled'] = F_samples(np.concatenate(a, 1))
    for name, lab in [('diff_clipfinal', 'diffusion, final-only clipping'), ('diff_clipnone', 'diffusion, no clipping')] + \
                     [(f'diff_cap{str(p).replace(".", "p")}', f'diffusion, cap {p}th pct') for p in [99.5, 99.9, 99.99, 100.0]]:
        a = stack(name, k) if k in ('calib', 'test') else []
        if a: M[f'{lab}, 3 seeds pooled'] = F_samples(np.concatenate(a, 1))
    return M
MS = {k: build_methods(k) for k in ['select', 'calib', 'test', 'test_broad']}
print('methods on test:', list(MS['test']))

## 4. Strongest baseline
Lowest mean quantile loss on select days among the probabilistic CNN baselines.

In [ ]:
BASELINES = [m for m in MS['select'] if m.startswith(('MSE CNN', 'Gaussian CNN', 'quantile CNN'))]
sel = pd.DataFrame([dict(method=m, select_quantile_loss=qloss_w(MS['select'][m], Y['select']).mean(),
                         select_crps=(np.nan if crps_w(MS['select'][m], Y['select']) is None else crps_w(MS['select'][m], Y['select']).mean()))
                    for m in BASELINES]).sort_values('select_quantile_loss')
print(sel.round(4).to_string(index=False))
BEST = sel.iloc[0].method
DIFF_MAIN = 'diffusion, image, 3 seeds pooled'
print(f'\nStrongest baseline (chosen on select days): {BEST}')
json.dump(dict(best_baseline=BEST, chosen_on='select days', table=sel.to_dict('records')), open(f'{P_DIR}/p4_baseline_choice.json', 'w'), indent=2, default=float)

## 5. Test scores

In [ ]:
def summary(k):
    rows = []
    for m, f in MS[k].items():
        if k == 'test_broad' and m not in ('smart persistence', BEST, DIFF_MAIN): continue
        y = Y[k]; mu = point(f); c = crps_w(f, y)
        r = dict(method=m, CRPS=np.nan if c is None else c.mean(), quantile_loss=qloss_w(f, y).mean(),
                 RMSE=np.sqrt(np.mean((mu - y) ** 2)), MAE=np.mean(np.abs(mu - y)))
        if k == 'test':
            fc = MS['calib'].get(m)
            for lev in LEVELS:
                for cal in (['none', 'split', 'adaptive'] if fc is not None else ['none']):
                    lo, hi = interval(f, fc, lev, k, cal)
                    r[f'cov{int(lev*100)}_{cal}'] = np.mean((y >= lo) & (y <= hi)) if not np.isnan(lo).all() else np.nan
                    r[f'width{int(lev*100)}_{cal}'] = np.mean(np.maximum(hi - lo, 0)) if not np.isnan(lo).all() else np.nan
                    r[f'empty{int(lev*100)}_{cal}'] = np.mean(hi < lo) if not np.isnan(lo).all() else np.nan
        rows.append(r)
    return pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
SUM = summary('test'); SUM.to_csv(f'{P_DIR}/p4_test_scores.csv', index=False)
SUM['family'] = SUM.method.str.replace(r', seed \d+$', '', regex=True)
single = SUM[SUM.method.str.contains(r', seed \d+$')]
SEEDTAB = single.groupby('family')[['CRPS', 'quantile_loss', 'RMSE', 'MAE', 'cov90_none', 'cov90_adaptive', 'width90_adaptive']].agg(['mean', 'std'])
SEEDTAB.to_csv(f'{P_DIR}/p4_seed_summary.csv'); print('Individual seeds (mean and std over 3 seeds):'); print(SEEDTAB.round(3).to_string()); print()
EMP = SUM[[c for c in SUM if c.startswith('empty')]]
print('max share of empty conformal intervals:', float(np.nanmax(EMP.values)) if EMP.size else 0.0)
print('All methods (single seeds and combined models):')
main_cols = ['method', 'CRPS', 'quantile_loss', 'RMSE', 'MAE', 'cov90_none', 'cov90_split', 'cov90_adaptive', 'width90_adaptive']
print(SUM[[c for c in main_cols if c in SUM]].round(3).to_string(index=False))

## 6. Paired bootstrap

In [ ]:
def day_sums(vals, days):
    g = pd.DataFrame({'d': days, 'v': vals, 'n': 1}).groupby('d').sum(); return g['v'].values, g['n'].values

def paired_boot(va, vb, days, kind='mean', n=N_BOOT, seed=0):
    '''Difference a - b of mean (or RMSE) per window, resampling whole days. Returns estimate and 95% interval.'''
    sa, nn = day_sums(va, days); sb, _ = day_sums(vb, days)
    idx = np.random.default_rng(seed).integers(0, len(nn), (n, len(nn)))
    A, B, N = sa[idx].sum(1), sb[idx].sum(1), nn[idx].sum(1)
    if kind == 'rmse': A, B = np.sqrt(A / N), np.sqrt(B / N); est = np.sqrt(va.mean()) - np.sqrt(vb.mean())
    else: A, B = A / N, B / N; est = va.mean() - vb.mean()
    d = A - B; rel = 1 - A / B
    base = np.sqrt(vb.mean()) if kind == 'rmse' else vb.mean()
    return dict(diff=est, diff_lo=np.percentile(d, 2.5), diff_hi=np.percentile(d, 97.5),
                rel_improvement=-est / base, rel_lo=np.percentile(rel, 2.5), rel_hi=np.percentile(rel, 97.5))

def per_window(m, k='test', lev=0.9):
    f, y = MS[k][m], Y[k]; c = crps_w(f, y); mu = point(f)
    out = dict(qloss=qloss_w(f, y), sqerr=(mu - y) ** 2, abserr=np.abs(mu - y))
    if c is not None: out['crps'] = c
    if k == 'test' and m in MS['calib']:
        lo, hi = interval(f, MS['calib'][m], lev, k, 'adaptive'); out['cov90_adaptive'] = ((y >= lo) & (y <= hi)).astype(float)
    return out

PAIRS = [(DIFF_MAIN, BEST), (DIFF_MAIN, 'smart persistence'), (DIFF_MAIN, 'Gaussian CNN mixture, 3 seeds'), (DIFF_MAIN, 'quantile CNN, 3 seeds averaged'), (DIFF_MAIN, 'diffusion, no image, 3 seeds pooled'),
         (DIFF_MAIN, 'diffusion, final-only clipping, 3 seeds pooled'), (DIFF_MAIN, 'diffusion, no clipping, 3 seeds pooled'),
         (BEST, 'smart persistence')]
PAIRS = list(dict.fromkeys(PAIRS))
rows = []
for a, b in PAIRS:
    if a not in MS['test'] or b not in MS['test']: continue
    pa, pb = per_window(a), per_window(b)
    for metric, kind in [('crps', 'mean'), ('qloss', 'mean'), ('sqerr', 'rmse'), ('abserr', 'mean'), ('cov90_adaptive', 'mean')]:
        if metric in pa and metric in pb:
            r = paired_boot(pa[metric], pb[metric], DAYS['test'], kind)
            rows.append(dict(a=a, b=b, metric={'sqerr': 'RMSE', 'abserr': 'MAE', 'crps': 'CRPS', 'qloss': 'quantile loss', 'cov90_adaptive': 'coverage 90% (adaptive)'}[metric], **r))
BOOT = pd.DataFrame(rows); BOOT.to_csv(f'{P_DIR}/p4_bootstrap.csv', index=False)
print('diff = a - b, rel_improvement = 1 - a/b')
print(BOOT.round(4).to_string(index=False))

## 7. Results by weather and on the broad test set

In [ ]:
def day_classes(D):
    day = D['zen'] < 80
    df = pd.DataFrame({'d': D['ld'][day], 'pv': D['pv'][day], 'cs': D['cs'][day], 'k': D['k'][day]}); g = df.groupby('d')
    mean_k = g.pv.sum() / g.cs.sum().clip(lower=1e-6)
    var_k = g.k.apply(lambda s: np.abs(np.diff(s.values)).mean() if len(s) > 1 else np.nan)
    return pd.Series(np.where((var_k < 0.01) & (mean_k > 0.7), 'clear', np.where(mean_k < 0.45, 'overcast', 'partly cloudy')), index=mean_k.index)
KINDS = ['clear', 'partly cloudy', 'overcast']
WCLS = {k: pd.Series(DAYS[k]).map(day_classes(TE)).values for k in ['test', 'test_broad']}
rows = []
for m in [x for x in ['smart persistence', BEST, DIFF_MAIN, 'diffusion, no image, 3 seeds pooled'] if x in MS['test']]:
    pw = per_window(m)
    for kd in KINDS:
        mk = WCLS['test'] == kd
        if not mk.any(): continue
        rows.append(dict(method=m, weather=kd, windows=int(mk.sum()), days=len(set(DAYS['test'][mk])),
                         CRPS=pw['crps'][mk].mean() if 'crps' in pw else np.nan, quantile_loss=pw['qloss'][mk].mean(),
                         RMSE=np.sqrt(pw['sqerr'][mk].mean()), cov90_adaptive=pw.get('cov90_adaptive', np.full(mk.size, np.nan))[mk].mean()))
WEA = pd.DataFrame(rows); WEA.to_csv(f'{P_DIR}/p4_by_weather.csv', index=False); print(WEA.round(3).to_string(index=False))

BROAD = summary('test_broad'); BROAD.to_csv(f'{P_DIR}/p4_broad_scores.csv', index=False)
print(f'\nBroad test set ({len(W["test_broad"])} windows vs {len(W["test"])} in the main set):'); print(BROAD.round(3).to_string(index=False))
rows = []
for a, b in [(DIFF_MAIN, BEST), (DIFF_MAIN, 'smart persistence')]:
    if a in MS['test_broad'] and b in MS['test_broad']:
        pa, pb = per_window(a, 'test_broad'), per_window(b, 'test_broad')
        for metric, kind, lab in [('crps', 'mean', 'CRPS'), ('qloss', 'mean', 'quantile loss'), ('sqerr', 'rmse', 'RMSE')]:
            if metric in pa and metric in pb: rows.append(dict(a=a, b=b, metric=lab, **paired_boot(pa[metric], pb[metric], DAYS['test_broad'], kind)))
pd.DataFrame(rows).to_csv(f'{P_DIR}/p4_broad_bootstrap.csv', index=False); print(pd.DataFrame(rows).round(4).to_string(index=False))

## 8. Reserve sizing
Schedule S is smart persistence for every method, L is the method's lower bound (calibrated on calib days), and the reserve is R = max(S − L, 0). A shortfall occurs when actual PV + R < S.

In [ ]:
S_te = smart_base(TE, W['test']).astype(float); y = Y['test']
RM = list(dict.fromkeys([m for m in ['smart persistence', 'MSE CNN, 3 seeds (spread)', 'Gaussian CNN mixture, 3 seeds', 'quantile CNN, 3 seeds averaged',
                  BEST, DIFF_MAIN, 'diffusion, no image, 3 seeds pooled'] if m in MS['test']]))
rows, PERW = [], {}
for m in RM:
    for rel in RELIAB:
        L = lower_bound(MS['test'][m], MS['calib'][m], rel, 'test'); R = np.maximum(S_te - L, 0)
        short = (y + R < S_te).astype(float); deficit = np.maximum(S_te - y - R, 0)
        PERW[(m, rel)] = dict(reserve=R, short=short, deficit=deficit)
        r = dict(method=m, reliability=rel, shortfall_rate=short.mean(), mean_reserve_kw=R.mean(), mean_deficit_kw=deficit.mean())
        for kd in KINDS:
            mk = WCLS['test'] == kd
            if mk.any(): r[f'shortfall_{kd}'] = short[mk].mean(); r[f'reserve_{kd}'] = R[mk].mean()
        rows.append(r)
RES = pd.DataFrame(rows); RES.to_csv(f'{P_DIR}/p4_reserve.csv', index=False); print(RES.round(4).to_string(index=False))
rows = []
for rel in RELIAB:
    for b in list(dict.fromkeys([BEST, 'smart persistence', 'Gaussian CNN mixture, 3 seeds'])):
        if (DIFF_MAIN, rel) in PERW and (b, rel) in PERW:
            for q in ['reserve', 'short', 'deficit']:
                rows.append(dict(a=DIFF_MAIN, b=b, reliability=rel, quantity=q, **paired_boot(PERW[(DIFF_MAIN, rel)][q], PERW[(b, rel)][q], DAYS['test'])))
RBOOT = pd.DataFrame(rows); RBOOT.to_csv(f'{P_DIR}/p4_reserve_bootstrap.csv', index=False); print(RBOOT.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 4))
for m in RM:
    q = RES[RES.method == m]; ax.plot(100 * q.shortfall_rate, q.mean_reserve_kw, marker='o', label=m)
for rel in RELIAB: ax.axvline(100 * (1 - rel), color='0.8', lw=0.8, ls='--')
ax.set_xlabel('shortfall rate (%)'); ax.set_ylabel('mean reserve (kW)'); ax.set_title('Reserve vs shortfall (targets 90 / 95 / 97.5%)'); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(f'{P_DIR}/figs/p4_reserve_curve.png', dpi=150); plt.show()

## 9. Training curves

In [ ]:
rows = []
for f in sorted(glob.glob(f'{P_DIR}/ckpt/*.pt')):
    st = torch.load(f, map_location='cpu', weights_only=False); h = [v for _, v in st['hist']]
    if not h: continue
    b = int(np.argmin(h)); lim = DIFF['max_epochs'] if 'p2_' in f else TRAIN['max_epochs']
    rows.append(dict(model=os.path.basename(f)[:-3], epochs=len(h), budget=lim, hit_budget=len(h) >= lim, best_epoch=b + 1,
                     best_select_loss=min(h), gain_last10_pct=100 * (1 - min(h) / min(h[:max(1, len(h) - 10)]))))
CURVES = pd.DataFrame(rows); CURVES.to_csv(f'{P_DIR}/p4_training_curves.csv', index=False); print(CURVES.round(4).to_string(index=False))

## 10. Reliability diagram and example day

In [ ]:
lev_fine = [0.5, 0.8, 0.9, 0.95]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for m in [x for x in ['MSE CNN, 3 seeds (spread)', 'Gaussian CNN mixture, 3 seeds', 'quantile CNN, 3 seeds averaged', DIFF_MAIN, 'diffusion, no image, 3 seeds pooled'] if x in MS['test']]:
    cov = [np.mean((y >= lo) & (y <= hi)) for lo, hi in (interval(MS['test'][m], None, l, 'test', 'none') for l in lev_fine)]
    axes[0].plot(lev_fine, cov, marker='o', label=m)
axes[0].plot([0.45, 1], [0.45, 1], '--', color='0.5', lw=1, label='perfect calibration')
axes[0].set_xlabel('nominal central interval level'); axes[0].set_ylabel('observed coverage (test)'); axes[0].set_title('Before recalibration'); axes[0].legend(fontsize=9)
d = pd.Series(DAYS['test']); top = pd.Series(np.abs(np.diff(y, prepend=y[0]))).groupby(d.values).mean().idxmax()
mk = DAYS['test'] == top; ts = TE['ts'][W['test'][mk] + HORIZON]
lo, hi = interval(MS['test'][DIFF_MAIN], MS['calib'][DIFF_MAIN], 0.9, 'test', 'adaptive')
axes[1].fill_between(ts, lo[mk], hi[mk], color='C1', alpha=0.25, lw=0, label='diffusion 90% interval (adaptive)')
axes[1].plot(ts, y[mk], 'k', lw=1, label='actual PV'); axes[1].plot(ts, S_te[mk], '--', color='0.5', lw=1, label='smart persistence')
axes[1].plot(ts, point(MS['test'][DIFF_MAIN])[mk], color='C1', lw=1, label='diffusion mean')
axes[1].set_ylabel('kW'); axes[1].set_title(f'Most variable test day ({top}), final pooled model'); axes[1].legend(fontsize=9)
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(f'{P_DIR}/figs/p4_reliability_and_example.png', dpi=150); plt.show()

## 11. Per-forecast table and training cost

In [ ]:
PF = pd.DataFrame({'issue_time_utc': TE['ts'][W['test']], 'target_time_utc': TE['ts'][W['test'] + HORIZON], 'local_day': DAYS['test'],
                   'weather': WCLS['test'], 'actual_kw': y, 'smart_persistence_kw': S_te})
for m in [x for x in [BEST, DIFF_MAIN, 'diffusion, no image, 3 seeds pooled'] if x in MS['test']]:
    f = MS['test'][m]; key = m.split(',')[0].replace(' ', '_') + ('_noimg' if 'no image' in m else '')
    PF[f'{key}_mean'] = point(f); PF[f'{key}_q05'] = quantile(f, 0.05); PF[f'{key}_q95'] = quantile(f, 0.95)
PF.to_csv(f'{P_DIR}/p4_per_forecast_test.csv.gz', index=False); print('per-forecast table:', PF.shape)
cps = [pd.read_csv(p) for p in [f'{P_DIR}/p1_training_cost.csv', f'{P_DIR}/p2_training_cost.csv'] if os.path.exists(p)]
if cps:
    cost = pd.concat(cps); cost.to_csv(f'{P_DIR}/p4_training_cost.csv', index=False)
    print(cost.groupby('model')[['epochs', 'train_minutes']].mean().round(1))

def g(df, a, b, met):
    r = df[(df.a == a) & (df.b == b) & (df.metric == met)]
    return None if r.empty else r.iloc[0]
print('\n===== SUMMARY =====')
print('Strongest CNN baseline (select days):', BEST)
for b in [BEST, 'diffusion, no image, 3 seeds pooled', 'diffusion, final-only clipping, 3 seeds pooled']:
    r = g(BOOT, DIFF_MAIN, b, 'CRPS') if b != BEST or crps_w(MS['test'][BEST], y) is not None else g(BOOT, DIFF_MAIN, b, 'quantile loss')
    if r is not None:
        verdict = 'diffusion better' if r.diff_hi < 0 else ('diffusion worse' if r.diff_lo > 0 else 'no difference detected')
        print(f'{DIFF_MAIN} vs {b} ({r.metric}): {100*r.rel_improvement:+.1f}% [{100*r.rel_lo:+.1f}, {100*r.rel_hi:+.1f}] -> {verdict}')
print('\nAll outputs are in MyDrive/UROP/review/ (p4_*.csv, p4_baseline_choice.json, figs/p4_*.png).')